# Gaze → video alignment (multi-video, one session)

One eye-tracking file for the whole session, **several non-overlapping videos** that
together cover it. Each sample's video frame comes straight from the export's own
columns — no `creation_time` bridging, no hand-read `START_TRUE` / `AFTER_TRUE` anchors.

| column | meaning |
|---|---|
| `varjo_recording_active` | `1` while a video is recording, else `0` |
| `is_varjo_record_start` | `1` on the exact sample where a video starts |
| `varjo_recording_index` | **which** recording a sample belongs to (used when present) |
| `fixation_target` | the object of the current fixation (NA outside a fixation) |
| `relative_to_video_first_frame_timestamp` | ns since **that** video's frame 0 (NaN outside a recording) |

Inside a recording, `video_frame_index = floor(relative_to_video_first_frame_timestamp * 1e-9 * fps)`.

Pipeline: **load → probe videos → segment & match files → frame index → phase parts → renderer → render.**

## 0. Setup

In [1]:
import json
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

COL = dict(
    time          = "gaze_capture_time",
    unix_ms       = "raw_timestamp",
    rec_active    = "varjo_recording_active",
    rec_start     = "is_varjo_record_start",
    rec_index     = "varjo_recording_index",
    rel_to_frame0 = "relative_to_video_first_frame_timestamp",
    model         = "model_name",
    condition     = "condition_number",
    status        = "status",
    fix_target    = "fixation_target",
    event         = "event_type",
    # Candidate gaze projections; GAZE_SOURCE below picks which pair is used.
    gaze_x        = "gaze_projected_to_left_view_x",
    gaze_y        = "gaze_projected_to_left_view_y",
    left_x        = "left_projected_x",
    left_y        = "left_projected_y",
    right_x       = "right_projected_x",
    right_y       = "right_projected_y",
)

FIXATION_LABELS = ("fixation",)   # values of event_type that count as a fixation

TUTORIAL_MODEL = "TM"

STYLE = dict(
    dot_radius=18, dot_color=(0, 0, 255), dot_alpha=0.75,
    dot_outline=(255, 255, 255), dot_outline_th=2,
    box_gap=24, box_pad=14, box_alpha=0.55, min_box_w=120,
    font=cv2.FONT_HERSHEY_SIMPLEX, font_scale=1.0, font_thick=2,
    clamp_offscreen=True,
    pick="median",          # "median" | "centered" | "nearest"
    gaze_source="left_view",  # "left_view" | "left" | "right" | "mean"
    smooth_fixations=0,       # frames of rolling median inside a fixation; 0 = off
    scale_x=1.0, offset_x=0.0, scale_y=1.0, offset_y=0.0,
)


def load_session(csv_path):
    df = pd.read_parquet(csv_path)
    if COL["rel_to_frame0"] not in df.columns:
        for alt in ("relative_to_video_first_frame_timestep",):
            if alt in df.columns:
                df = df.rename(columns={alt: COL["rel_to_frame0"]})
                break
    needed = [COL["time"], COL["rec_active"], COL["rec_start"], COL["rel_to_frame0"]]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")
    return df.sort_values(COL["time"]).reset_index(drop=True)


def sample_rate_hz(df):
    """Median ET sampling rate, from the monotonic capture clock (ns)."""
    d = pd.to_numeric(df[COL["time"]], errors="coerce").diff()
    step_ns = d[d > 0].median()
    return float(1e9 / step_ns) if step_ns and np.isfinite(step_ns) else np.nan

## 1. Configure

Video order does not matter — files are matched to the data by creation timestamp.

In [2]:
CSV_PATH = "data/merged/998_synced_with_objects.parquet"      # whole-session ET file

VIDEO_PATHS = [
    "data/video/998_varjo_capture_15-57.mp4",
    "data/video/998_varjo_capture_15-58.mp4",
    "data/video/998_varjo_capture_16-03.mp4",
    "data/video/998_varjo_capture_16-07.mp4",
]

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load the session

Sorted by the monotonic capture clock — raw rows can be out of order. The sampling rate
is measured rather than assumed; it is what the coverage checks later on are built on.

In [3]:
def load_session(csv_path):
    df = pd.read_parquet(csv_path)
    if COL["rel_to_frame0"] not in df.columns:
        for alt in ("relative_to_video_first_frame_timestep",):
            if alt in df.columns:
                df = df.rename(columns={alt: COL["rel_to_frame0"]})
                break
    needed = [COL["time"], COL["rec_active"], COL["rec_start"], COL["rel_to_frame0"]]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")
    return df.sort_values(COL["time"]).reset_index(drop=True)


def sample_rate_hz(df):
    """Median ET sampling rate, from the monotonic capture clock (ns)."""
    d = pd.to_numeric(df[COL["time"]], errors="coerce").diff()
    step_ns = d[d > 0].median()
    return float(1e9 / step_ns) if step_ns and np.isfinite(step_ns) else np.nan


# --------------------------------------------------------------------------- probe


df = load_session(CSV_PATH)
HZ = sample_rate_hz(df)
print(f"{len(df):,} eye-tracking samples loaded at ~{HZ:.1f} Hz")

133,785 eye-tracking samples loaded at ~200.0 Hz


## 3. Probe the videos

fps, size, creation time and **frame count** via `ffprobe`, with OpenCV filling gaps.
The frame count matters: any computed frame index beyond it is proof of a bad alignment.

In [4]:
def _creation_ms(tags):
    ct = (tags or {}).get("creation_time")
    if not ct:
        return None
    dt = datetime.fromisoformat(ct.strip().replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.timestamp() * 1000.0


def probe_video(path):
    info = dict(path=path, fps=None, creation_ms=None, width=None, height=None,
                n_frames=None)
    if shutil.which("ffprobe"):
        out = subprocess.run(
            ["ffprobe", "-v", "quiet", "-print_format", "json",
             "-show_format", "-show_streams", path],
            capture_output=True, text=True).stdout
        meta = json.loads(out) if out.strip() else {}
        info["creation_ms"] = _creation_ms(meta.get("format", {}).get("tags"))
        for st in meta.get("streams", []):
            if st.get("codec_type") != "video":
                continue
            rate = st.get("avg_frame_rate") or st.get("r_frame_rate") or "0/0"
            num, _, den = rate.partition("/")
            den = den or "1"
            if float(den) != 0 and float(num) != 0:
                info["fps"] = float(num) / float(den)
            info["width"], info["height"] = st.get("width"), st.get("height")
            if info["creation_ms"] is None:
                info["creation_ms"] = _creation_ms(st.get("tags"))
            break
    # n_frames is always read here: it is the upper bound every frame index must
    # respect, and an index beyond it means the alignment is wrong.
    cap = cv2.VideoCapture(path)
    info["fps"]      = info["fps"]      or float(cap.get(cv2.CAP_PROP_FPS))
    info["width"]    = info["width"]    or int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    info["height"]   = info["height"]   or int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    info["n_frames"] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
    cap.release()
    return info


# ----------------------------------------------------------------------- segments


videos = [probe_video(p) for p in VIDEO_PATHS]
for v in videos:
    print(f"{Path(v['path']).name:38s} {v['fps']:.4f} fps  "
          f"{v['width']}x{v['height']}  {v['n_frames']} frames")

998_varjo_capture_15-57.mp4            30.0000 fps  2880x2720  1063 frames
998_varjo_capture_15-58.mp4            30.0000 fps  2880x2720  8810 frames
998_varjo_capture_16-03.mp4            30.0000 fps  2880x2720  5634 frames
998_varjo_capture_16-07.mp4            30.0000 fps  2880x2720  6195 frames


## 4. Split into per-video segments & match files

If the export has `varjo_recording_index`, that column *states* which recording each
sample belongs to and is used directly. The `is_varjo_record_start` cumulative sum is
kept as a fallback and as a cross-check — if the two disagree on many rows, the
inferred version was wrong and it is worth finding out why.

`start_gap_s` (recording start − file creation) should be small and consistent across
all rows. A wildly different value in one row means that video is paired with the wrong
segment, and every clip from it will be misaligned.

In [5]:
def segments_from_flags(df):
    """Fallback: cumulative sum of is_varjo_record_start, masked to active rows."""
    active = df[COL["rec_active"]].fillna(0).astype(int).eq(1)
    seg = df[COL["rec_start"]].fillna(0).astype(int).cumsum().where(active)
    return seg.astype("Int64")


def assign_segments(df, verbose=True):
    """Segment id per row (Int64, NA when no video is recording), 1-based & chronological.

    Prefers the explicit `varjo_recording_index` column when the export provides it;
    that column states which recording a sample belongs to, so no inference is needed.
    Falls back to the is_varjo_record_start cumulative sum otherwise. When both are
    available they are compared, because a disagreement means the fallback would have
    mis-assigned rows.
    """
    flag_seg = segments_from_flags(df)

    if COL["rec_index"] not in df.columns:
        if verbose:
            print(f"[segments] '{COL['rec_index']}' not found - using "
                  f"{COL['rec_start']} cumulative sum")
        return flag_seg

    active = df[COL["rec_active"]].fillna(0).astype(int).eq(1)
    idx = pd.to_numeric(df[COL["rec_index"]], errors="coerce").where(active)
    present = sorted(idx.dropna().unique())
    if not present:
        if verbose:
            print(f"[segments] '{COL['rec_index']}' is empty - falling back to flags")
        return flag_seg

    # Normalise whatever labels the column uses to 1..N in chronological order.
    remap = {v: i + 1 for i, v in enumerate(present)}
    seg = idx.map(remap).astype("Int64")

    if verbose:
        agree = (seg == flag_seg) | (seg.isna() & flag_seg.isna())
        n_dis = int((~agree).sum())
        print(f"[segments] {len(present)} recordings from '{COL['rec_index']}' "
              f"(values {present} -> 1..{len(present)})")
        print(f"[segments] rows where the {COL['rec_start']} fallback disagrees: {n_dis:,}")
    return seg


def summarise_segments(df, seg):
    rows = []
    for sid, g in df.groupby(seg):
        rows.append(dict(
            segment=int(sid),
            n_samples=len(g),
            start_unix_ms=float(g[COL["unix_ms"]].iloc[0]),
            end_unix_ms=float(g[COL["unix_ms"]].iloc[-1]),
        ))
    return pd.DataFrame(rows).sort_values("segment").reset_index(drop=True)


def map_segments_to_videos(segments, videos, verbose=True):
    if all(v["creation_ms"] for v in videos):
        vids = sorted(videos, key=lambda v: v["creation_ms"])
    else:
        vids = sorted(videos, key=lambda v: v["path"])
    if len(vids) != len(segments):
        print(f"[warn] {len(segments)} recording segments but {len(vids)} videos - "
              f"the pairing below is almost certainly shifted; fix the file list first")

    mapping, check = {}, []
    for (_, seg), vid in zip(segments.iterrows(), vids):
        mapping[seg["segment"]] = vid
        gap = ((seg["start_unix_ms"] - vid["creation_ms"]) / 1000.0
               if vid["creation_ms"] else np.nan)
        check.append(dict(segment=seg["segment"], video=Path(vid["path"]).name,
                          fps=round(vid["fps"], 4), n_samples=seg["n_samples"],
                          start_gap_s=round(gap, 1)))
    if verbose:
        print(pd.DataFrame(check).to_string(index=False))
    return mapping


# -------------------------------------------------------------------- frame index


seg       = assign_segments(df)
segments  = summarise_segments(df, seg)
seg2video = map_segments_to_videos(segments, videos)

[segments] 4 recordings from 'varjo_recording_index' (values [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)] -> 1..4)
[segments] rows where the is_varjo_record_start fallback disagrees: 0
 segment                       video  fps  n_samples  start_gap_s
     1.0 998_varjo_capture_15-57.mp4 30.0       60.0         27.6
     2.0 998_varjo_capture_15-58.mp4 30.0    53913.0          0.4
     3.0 998_varjo_capture_16-03.mp4 30.0    34375.0          0.4
     4.0 998_varjo_capture_16-07.mp4 30.0    38561.0          0.3


## 5. Video frame index for every sample

Read the table as: for the samples belonging to this **video**, the earliest and latest
frame **of that video** they land on. Frame 0 is always that video's own first frame —
the numbers are not session-wide and not relative to a phase.

* `n_samples` — eye-tracking rows recorded while this video was running
* `span_frames` — `video_frame_last − video_frame_first + 1`
* `frames_with_data` / `coverage` — how much of that span actually has gaze on it.
  Well below 1.0 means the recording had holes; the clip will show the dot vanishing.

In [6]:
def frame_index_from_rel(df, seg, seg2video, clamp_negative=True, verbose=True):
    """floor(relative_to_video_first_frame_timestamp * 1e-9 * fps), per segment.

    A sample can carry a slightly negative offset when recording is flagged a few ms
    before the video's frame 0; those land on frame -1 and are clamped to 0 so they
    are not silently dropped by the renderer.
    """
    fps = pd.Series(np.nan, index=df.index)
    for sid, vid in seg2video.items():
        fps[seg == sid] = vid["fps"]
    rel_s = pd.to_numeric(df[COL["rel_to_frame0"]], errors="coerce") * 1e-9
    fi = np.floor(rel_s * fps)
    fi = fi.where(seg.notna())                    # only rows inside a recording
    if clamp_negative:
        n_neg = int((fi < 0).sum())
        if n_neg and verbose:
            print(f"[frames] {n_neg} sample(s) just before frame 0 clamped to 0")
        fi = fi.clip(lower=0)
    return fi.astype("Int64")


def summarise_frame_index(df, seg2video):
    """Per video: how much of the frame span the ET data actually covers."""
    rows = []
    for sid, vid in seg2video.items():
        fi = df.loc[df["segment"] == sid, "video_frame_index"].dropna()
        if fi.empty:
            rows.append(dict(segment=sid, video=Path(vid["path"]).name,
                             fps=round(vid["fps"], 4), n_samples=0))
            continue
        span = int(fi.max() - fi.min() + 1)
        covered = int(fi.nunique())
        rows.append(dict(
            segment=sid, video=Path(vid["path"]).name, fps=round(vid["fps"], 4),
            n_samples=len(fi),
            video_frame_first=int(fi.min()), video_frame_last=int(fi.max()),
            span_frames=span, frames_with_data=covered,
            coverage=round(covered / span, 3),
        ))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------------- phases


df["segment"]           = seg
df["video_frame_index"] = frame_index_from_rel(df, seg, seg2video)

print()
print(summarise_frame_index(df, seg2video).to_string(index=False))

[frames] 3 sample(s) just before frame 0 clamped to 0

 segment                       video  fps  n_samples  video_frame_first  video_frame_last  span_frames  frames_with_data  coverage
     1.0 998_varjo_capture_15-57.mp4 30.0         60                816               825           10                10     1.000
     2.0 998_varjo_capture_15-58.mp4 30.0      53913                  0              8810         8811              8282     0.940
     3.0 998_varjo_capture_16-03.mp4 30.0      34375                  0              5633         5634              5276     0.936
     4.0 998_varjo_capture_16-07.mp4 30.0      38561                  0              6194         6195              5891     0.951


## 6. Building phases, split per video

Each contiguous run of one real `model_name` (tutorial build `TM` excluded) is a phase.
A phase that straddles a recording boundary produces **two rows** here, one per video,
each with its own frame range — these are rendered as two clips.

Columns mean the same as in section 5, so `video_frame_first` / `video_frame_last` are
this part's first and last frame *within its own video*. `n_samples_in_gap` counts rows
of the phase that fall outside any recording: they have no video to be drawn on and are
skipped, which is the honest answer rather than a dot in the wrong place.

In [7]:
def label_building_phases(df):
    """Int64 phase id per row (NA outside a phase); tutorial build excluded.

    A new phase starts whenever `model_name` changes, not only when it goes to/from
    NaN - otherwise two phases that follow each other back to back with no idle rows
    in between would be merged into one run under the first model's name.
    """
    model = df[COL["model"]]
    is_real = model.notna() & model.ne(TUTORIAL_MODEL)
    key = model.fillna("\x00none").astype(str)
    run = (key != key.shift()).cumsum().where(is_real)
    codes = {r: i + 1 for i, r in enumerate(sorted(run.dropna().unique()))}
    return run.map(codes).astype("Int64")


def phase_parts(df):
    """One row per (building phase x video) pair - the unit that can be rendered.

    A phase is a contiguous run of one model_name. Recording can start or stop in the
    middle of such a run, so a phase may be spread over two videos (or partly fall in
    the gap between them). Summarising a phase as a single video with a single
    min/max frame therefore mixes frame numbers from different videos. Splitting by
    (phase, segment) keeps every row's frame number in the coordinate system of the
    video it was actually measured in.
    """
    rows = []
    for pid, g in df.groupby("phase"):
        n_total = len(g)
        n_gap = int(g["segment"].isna().sum())
        for sid, gs in g.groupby("segment"):
            frames = gs["video_frame_index"].dropna()
            if frames.empty:
                continue
            span = int(frames.max() - frames.min() + 1)
            cond = gs[COL["condition"]].iloc[0] if COL["condition"] in gs else pd.NA
            rows.append(dict(
                phase=int(pid),
                model_name=gs[COL["model"]].iloc[0],
                segment=int(sid),
                condition=int(cond) if pd.notna(cond) else pd.NA,
                video_frame_first=int(frames.min()),
                video_frame_last=int(frames.max()),
                span_frames=span,
                n_samples=len(gs),
                frames_with_data=int(frames.nunique()),
                coverage=round(frames.nunique() / span, 3),
                phase_n_samples=n_total,
                n_samples_in_gap=n_gap,
            ))
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["n_parts"] = out.groupby("phase")["phase"].transform("size")
    return out.sort_values(["phase", "segment"]).reset_index(drop=True)


def warn_about_parts(parts, min_coverage=0.7):
    """Print the phases that are split across videos or thinly covered."""
    if parts.empty:
        print("no building phases found")
        return
    split = parts[parts["n_parts"] > 1]
    if len(split):
        print("phases split across more than one video (rendered as separate clips):")
        print(split[["phase", "model_name", "segment", "video_frame_first",
                     "video_frame_last", "n_samples"]].to_string(index=False))
    thin = parts[parts["coverage"] < min_coverage]
    if len(thin):
        print(f"\nparts covering < {min_coverage:.0%} of their frame span "
              f"(gaps in the recording - check before trusting the clip):")
        print(thin[["phase", "model_name", "segment", "span_frames",
                    "frames_with_data", "coverage"]].to_string(index=False))
    gapped = parts[parts["n_samples_in_gap"] > 0].drop_duplicates("phase")
    if len(gapped):
        print("\nphases with samples outside any recording (no video to draw on):")
        print(gapped[["phase", "model_name", "phase_n_samples",
                      "n_samples_in_gap"]].to_string(index=False))


# ----------------------------------------------------------------------- renderer


df["phase"] = label_building_phases(df)
parts = phase_parts(df)

cols = ["phase", "model_name", "segment", "condition", "video_frame_first",
        "video_frame_last", "span_frames", "n_samples", "coverage", "n_parts"]
print(parts[cols].to_string(index=False))
print()
warn_about_parts(parts)

 phase model_name  segment  condition  video_frame_first  video_frame_last  span_frames  n_samples  coverage  n_parts
     1      C1M3F        2          1                617              1127          511       3079     0.941        1
     2      C3M2A        2          1               1448              8492         7045      42880     0.936        1
     3      C1M6A        2          1               8781              8810           30        169     0.900        2
     3      C1M6A        3          1                  0                62           63        382     0.937        2
     4      C2M6A        3          1                440              5299         4860      29371     0.930        1
     5      C2M5F        3          1               5611              5633           23        123     0.870        2
     5      C2M5F        4          1                  0              2533         2534      15642     0.945        2
     6      C3M5F        4          1               2812

## 7. Overlay renderer

Per frame: one gaze pixel and — **only while a fixation is running** — the object from
`fixation_target`. On a saccade frame there is no fixated object, so no box is drawn at
all rather than one carrying the previous fixation's target forward. The video is
decoded once, frames written straight out.

`STYLE["pick"]` decides which of the ~7 samples on a frame becomes the dot:

| `pick` | window | behaviour |
|---|---|---|
| `"median"` | `[N/fps, (N+1)/fps)` | averages tracker noise, effective moment half a frame **after** the frame's timestamp |
| `"centered"` | `[(N−0.5)/fps, (N+0.5)/fps)` | same averaging, centred on the frame's timestamp |
| `"nearest"` | closest single sample to `N/fps` | tracks fast movement exactly, full single-sample noise |

`STYLE["gaze_source"]` decides **which projection** is used — see section 10.
`STYLE["smooth_fixations"]` optionally applies a rolling median across frames, reset at
every event boundary so a saccade is never smoothed into its neighbours.

In [8]:
def _mode(s):
    s = s[s.notna() & (s.astype(str).str.strip() != "")]
    m = s.mode()
    return m.iloc[0] if not m.empty else pd.NA


def _fixation_blocks(evt):
    """Block id per frame; consecutive frames of the same event_type share one id."""
    e = evt.astype(str)
    return (e != e.shift()).cumsum()


def _is_fixation(evt):
    return evt.astype(str).str.lower().isin([v.lower() for v in FIXATION_LABELS])


def _fixation_label(t, has_evt):
    """Object label per frame - only while a fixation is running, never carried over.

    A saccade frame gets NA and the renderer then draws no box at all. This is
    deliberately different from carrying the previous fixation's target forward: during
    a saccade there is no fixated object, so any label shown would be a claim the data
    does not make.
    """
    if not has_evt:
        # No event column: fixation_target is itself only defined during fixations,
        # so use it as-is rather than inventing a fixation/saccade split.
        return t["obj"]
    label = pd.Series(pd.NA, index=t.index, dtype=object)
    fix = _is_fixation(t["evt"])
    for _, idx in t.groupby(_fixation_blocks(t["evt"])).groups.items():
        if fix.loc[idx].iloc[0]:
            label.loc[idx] = _mode(t.loc[idx, "obj"])
    return label


def gaze_ndc(g, source):
    """The chosen gaze projection as an (x, y) pair in normalised device coordinates.

    "left_view" is whatever the export projected into the left eye's view; "left" and
    "right" are the single-eye projections; "mean" averages them, which cancels the
    horizontal parallax between the two eyes and approximates a cyclopean projection.
    """
    def pair(kx, ky):
        for k in (kx, ky):
            if COL[k] not in g.columns:
                raise KeyError(f"gaze_source={source!r} needs column {COL[k]!r}")
        return (pd.to_numeric(g[COL[kx]], errors="coerce"),
                pd.to_numeric(g[COL[ky]], errors="coerce"))

    if source == "left_view":
        return pair("gaze_x", "gaze_y")
    if source == "left":
        return pair("left_x", "left_y")
    if source == "right":
        return pair("right_x", "right_y")
    if source == "mean":
        lx, ly = pair("left_x", "left_y")
        rx, ry = pair("right_x", "right_y")
        # mean of the two eyes; falls back to whichever eye is present on a sample
        return (pd.concat([lx, rx], axis=1).mean(axis=1, skipna=True),
                pd.concat([ly, ry], axis=1).mean(axis=1, skipna=True))
    raise ValueError(f"unknown gaze_source {source!r}")


def _smooth_within_fixations(t, n):
    """Rolling median of the dot position, reset at every event boundary.

    Smoothing across a saccade would drag the dot along behind the eye; smoothing
    inside a fixation only removes tracker jitter, which is the intent.
    """
    if not n or n < 2 or "evt" not in t:
        return t
    blocks = _fixation_blocks(t["evt"])
    for ax in ("px", "py"):
        t[ax] = (t.groupby(blocks)[ax]
                  .transform(lambda s: s.rolling(n, center=True, min_periods=1).median()))
    return t


def per_frame_table(rows, W, H, fps, style=STYLE, pick=None, pick_source=None):
    """One dot + one label per video frame.

    `pick` decides which of the ~fps/rate samples that fall on a frame becomes the dot:

    * "median"   - median over the interval the frame is on screen, [N/fps, (N+1)/fps).
                   Averages away tracker noise, but its effective time is the middle of
                   that interval, i.e. half a frame later than the frame's own timestamp.
    * "centered" - median over [(N-0.5)/fps, (N+0.5)/fps), i.e. the same averaging but
                   centred on the frame's timestamp. Removes the half-frame lead.
    * "nearest"  - the single sample closest in time to the frame's timestamp N/fps.
                   Follows fast movement exactly; carries the full single-sample noise.

    "nearest" is not the same as "first sample in the bin": the closest sample to N/fps
    is often a few ms *before* it and would be binned into frame N-1, so the match is
    done on the timestamps themselves rather than inside the existing bins.
    """
    pick = pick or style.get("pick", "median")
    g = rows[rows[COL["status"]].astype(str).eq("Valid")].copy()
    g = g.dropna(subset=["video_frame_index"])
    if g.empty:
        return g.assign(px=[], py=[], obj=[], label=[])

    x, y = gaze_ndc(g, pick_source or style.get("gaze_source", "left_view"))
    g["px"] = ((x + 1.0) * 0.5 * W) * style["scale_x"] + style["offset_x"]
    g["py"] = ((1.0 - y) * 0.5 * H) * style["scale_y"] + style["offset_y"]
    g["rel_s"] = pd.to_numeric(g[COL["rel_to_frame0"]], errors="coerce") * 1e-9

    has_evt = COL["event"] in g.columns

    if pick == "nearest":
        t = _pick_nearest(g, fps, has_evt)
    else:
        # "median" keeps the existing floor() bins; "centered" shifts them half a frame.
        key = (g["video_frame_index"].astype(int) if pick == "median"
               else np.floor(g["rel_s"] * fps + 0.5).astype(int))
        agg = {"px": ("px", "median"), "py": ("py", "median"),
               "obj": (COL["fix_target"], _mode)}
        if has_evt:
            agg["evt"] = (COL["event"], _mode)
        t = g.groupby(key).agg(**agg).sort_index()

    if t.empty:
        return t
    t.index.name = "video_frame_index"
    t = _smooth_within_fixations(t, style.get("smooth_fixations", 0))
    t["label"] = _fixation_label(t, has_evt)
    return t


def _pick_nearest(g, fps, has_evt, tolerance_frames=1.0):
    """For every frame in range, the sample whose timestamp is closest to N/fps."""
    cols = ["rel_s", "px", "py", COL["fix_target"]] + ([COL["event"]] if has_evt else [])
    s = g[cols].dropna(subset=["rel_s"]).sort_values("rel_s")
    lo = int(np.floor(s["rel_s"].iloc[0] * fps))
    hi = int(np.floor(s["rel_s"].iloc[-1] * fps))
    frames = pd.DataFrame({"frame": np.arange(max(lo, 0), hi + 1)})
    frames["rel_s"] = frames["frame"] / fps

    m = pd.merge_asof(frames, s, on="rel_s", direction="nearest",
                      tolerance=tolerance_frames / fps)
    m = m.dropna(subset=["px", "py"]).set_index("frame")
    out = m[["px", "py"]].copy()
    out["obj"] = m[COL["fix_target"]]
    if has_evt:
        out["evt"] = m[COL["event"]]
    return out


def _draw(img, px, py, label, style=STYLE):
    H, W = img.shape[:2]
    if pd.isna(px) or pd.isna(py):
        return img
    if style["clamp_offscreen"]:
        px, py = float(np.clip(px, 0, W - 1)), float(np.clip(py, 0, H - 1))
    elif not (0 <= px < W and 0 <= py < H):
        return img
    cx, cy, R = int(round(px)), int(round(py)), style["dot_radius"]

    ov = img.copy()
    cv2.circle(ov, (cx, cy), R, style["dot_color"], -1, cv2.LINE_AA)
    if style["dot_outline_th"]:
        cv2.circle(ov, (cx, cy), R, style["dot_outline"], style["dot_outline_th"], cv2.LINE_AA)
    cv2.addWeighted(ov, style["dot_alpha"], img, 1 - style["dot_alpha"], 0, img)

    # No label -> no box. Outside a fixation there is no fixated object, so an empty
    # box would still assert "something is being looked at". Draw the dot only.
    text = "" if (label is None or pd.isna(label)) else str(label).strip()
    if not text:
        return img

    F, S, TH, PAD = style["font"], style["font_scale"], style["font_thick"], style["box_pad"]
    (tw, _), _  = cv2.getTextSize(text, F, S, TH)
    (_, th), bl = cv2.getTextSize("Ag", F, S, TH)
    bw = max(style["min_box_w"], tw + 2 * PAD)
    bh = th + bl + 2 * PAD

    x1 = cx + R + style["box_gap"]
    if x1 + bw > W:
        x1 = cx - R - style["box_gap"] - bw
    x1 = int(np.clip(x1, 0, W - bw))
    y1 = int(np.clip(cy - bh // 2, 0, H - bh))

    bo = img.copy()
    cv2.rectangle(bo, (x1, y1), (x1 + bw, y1 + bh), (0, 0, 0), -1)
    cv2.addWeighted(bo, style["box_alpha"], img, 1 - style["box_alpha"], 0, img)
    cv2.rectangle(img, (x1, y1), (x1 + bw, y1 + bh), (255, 255, 255), 2)
    cv2.putText(img, text, (x1 + PAD, y1 + PAD + th), F, S, (255, 255, 255), TH, cv2.LINE_AA)
    return img


def render(rows, segment_id, seg2video, out_path, frame_range=None, style=STYLE):
    """Overlay gaze dot + label for one (phase, video) part.

    `rows` must contain samples from a single segment: frame numbers are per-video,
    so mixing segments would paint one video's gaze onto another's footage. The
    assertion below makes that impossible rather than merely unlikely.
    """
    segs = rows["segment"].dropna().unique()
    if len(segs) > 1:
        raise ValueError(f"rows span segments {sorted(segs)} - render one segment at a time")
    if len(segs) == 1 and int(segs[0]) != int(segment_id):
        raise ValueError(f"rows belong to segment {int(segs[0])}, not {segment_id}")

    video = seg2video[int(segment_id)]
    W, H, fps = video["width"], video["height"], video["fps"]
    table = per_frame_table(rows, W, H, fps, style)
    if table.empty:
        print(f"[skip] {Path(out_path).name}: no valid gaze frames")
        return
    look = {int(f): (r["px"], r["py"], r["label"]) for f, r in table.iterrows()}

    lo = int(table.index.min()) if frame_range is None else int(frame_range[0])
    hi = int(table.index.max()) if frame_range is None else int(frame_range[1])
    n_frames = video.get("n_frames")
    if n_frames and hi >= n_frames:
        print(f"[warn] {Path(out_path).name}: frame {hi} requested but "
              f"{Path(video['path']).name} only has {n_frames} - alignment is off "
              f"for this part")

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(video["path"])
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    i = written = 0
    while i <= hi:
        if not cap.grab():
            break
        if i >= lo:
            ok, frame = cap.retrieve()
            if not ok:
                break
            if i in look:
                _draw(frame, *look[i], style=style)
            writer.write(frame)
            written += 1
        i += 1
    cap.release()
    writer.release()
    if written == 0:
        print(f"[warn] {Path(out_path).name}: nothing written - frames {lo}..{hi} "
              f"are past the end of {Path(video['path']).name}")
    else:
        print(f"wrote {out_path}  (frames {lo}..{hi}, {written} written, "
              f"{len(look)} with gaze)")

## 8. Render — one clip per phase part

Rows are filtered on **both** phase and segment. Output names carry the source video, so
the two clips of a split phase stay distinguishable.

In [10]:
for _, p in parts.iterrows():
    sid  = int(p["segment"])
    vid  = seg2video[sid]
    rows = df[(df["phase"] == p["phase"]) & (df["segment"] == sid)]
    tag  = f"_part{sid}" if p["n_parts"] > 1 else ""
    out  = OUTPUT_DIR / f"{Path(vid['path']).stem}_{p['model_name']}{tag}_gaze.mp4"
    render(rows, sid, seg2video, out)

wrote results\998_varjo_capture_15-58_C1M3F_gaze.mp4  (frames 617..1127, 511 written, 481 with gaze)
wrote results\998_varjo_capture_15-58_C3M2A_gaze.mp4  (frames 1448..8492, 7045 written, 6591 with gaze)
[warn] 998_varjo_capture_15-58_C1M6A_part2_gaze.mp4: frame 8810 requested but 998_varjo_capture_15-58.mp4 only has 8810 - alignment is off for this part
wrote results\998_varjo_capture_15-58_C1M6A_part2_gaze.mp4  (frames 8781..8810, 29 written, 27 with gaze)
wrote results\998_varjo_capture_16-03_C1M6A_part3_gaze.mp4  (frames 0..62, 63 written, 59 with gaze)
wrote results\998_varjo_capture_16-03_C2M6A_gaze.mp4  (frames 440..5299, 4860 written, 4518 with gaze)
wrote results\998_varjo_capture_16-03_C2M5F_part3_gaze.mp4  (frames 5611..5633, 23 written, 20 with gaze)
wrote results\998_varjo_capture_16-07_C2M5F_part4_gaze.mp4  (frames 0..2533, 2534 written, 2395 with gaze)
wrote results\998_varjo_capture_16-07_C3M5F_gaze.mp4  (frames 2812..5878, 3067 written, 2912 with gaze)


## 9. Render — a whole video (optional)

The full-length check: every sample of one recording against its own video. Slow, so
opt-in — but it is the quickest way to confirm the alignment holds end to end.

In [ ]:
RENDER_WHOLE_VIDEOS = False   # set True to render each full recording

if RENDER_WHOLE_VIDEOS:
    for sid, vid in seg2video.items():
        rows = df[df["segment"] == sid]
        out  = OUTPUT_DIR / f"{Path(vid['path']).stem}_gaze.mp4"
        render(rows, sid, seg2video, out)